## Random Classifier Tests on Erosion

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

# One-hot encode LIBDENS
df['LIBDENS'] = df['LIBDENS'].astype('category')
df = pd.get_dummies(df, columns=['LIBDENS'], drop_first=True)

# Define features
libdens_cols = [col for col in df.columns if col.startswith('LIBDENS_')]
cols = ['urban', 'wetland', 'pprl', 'pprl_ero', 'pop_surexposed', 'nvm_surexposed', 'inec', 'nb_ouvrages_erosion'] + libdens_cols
df = df.dropna(subset= ['decret_2024'])
X = df[cols]
# Convert decret_2024 to int, handling NaN values
y = df['decret_2024'].astype(int)

# Fill NA values
X['pop_surexposed'] = X['pop_surexposed'].fillna(1)
X['nvm_surexposed'] = X['nvm_surexposed'].fillna(1)
X['pprl'] = X['pprl'].fillna(0)
X['pprl_ero'] = X['pprl_ero'].fillna(0)

# Drop rows with other missing values
print("LIBDENS before dropping NAs:")
print(X[libdens_cols].sum())
X_clean = X.dropna()
y_clean = y[X.index.isin(X_clean.index)]
print("LIBDENS after dropping NAs:")
print(X_clean[libdens_cols].sum())

# Train-test split (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.3, random_state=42, stratify=y_clean
)

# Random Forest with class_weight
model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
f1 = f1_score(y_test, y_pred)
print("F1 score on test set:", f1)
print("\nClassification report:\n", classification_report(y_test, y_pred))


In [ ]:
import pandas as pd
import numpy as np

# Feature importances
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)

# Display
print("Top features:\n", feat_imp.head(10))

# Optional: Plot feature importances
feat_imp.head(10).plot(kind='barh')
plt.xlabel("Importance")
plt.title("Top 10 Feature Importances")
plt.gca().invert_yaxis()
plt.show()
